In [1]:


import os
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    brier_score_loss,
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

# ============================================================
# OPTIONAL LIBRARIES
# ============================================================
HAS_XGBOOST = True
HAS_CATBOOST = True

try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGBOOST = False

try:
    from catboost import CatBoostClassifier
except Exception:
    HAS_CATBOOST = False

# ============================================================
# USER CONFIG
# ============================================================
DATA_ROOT = r"C:\Users\shanmugam\shan work 1\data\dataset_saved_500_300_pca4to12"
DS = "balanced"
LABEL_COL = "CLNSIG"

PCA_LIST = [12]   # PCA-12 ONLY
SEEDS = [0, 1, 2, 3, 4]
THRESH = 0.5

OUT_DIR = r"C:\Users\shanmugam\shan work 1\New folder\new try\output"
os.makedirs(OUT_DIR, exist_ok=True)

print("Results will be saved in:")
print(OUT_DIR)

# ============================================================
# LOAD DATA
# ============================================================
def load_train_test(root, ds, k):
    pdir = os.path.join(root, ds, f"pca_{k}")
    tr_path = os.path.join(pdir, "train.csv")
    te_path = os.path.join(pdir, "test.csv")

    if not (os.path.exists(tr_path) and os.path.exists(te_path)):
        raise FileNotFoundError(f"Missing: {tr_path} or {te_path}")

    tr = pd.read_csv(tr_path)
    te = pd.read_csv(te_path)

    if LABEL_COL not in tr.columns or LABEL_COL not in te.columns:
        raise ValueError(f"'{LABEL_COL}' missing in {ds}/pca_{k}")

    Xtr = tr.drop(columns=[LABEL_COL]).values.astype(float)
    ytr = tr[LABEL_COL].values.astype(int)

    Xte = te.drop(columns=[LABEL_COL]).values.astype(float)
    yte = te[LABEL_COL].values.astype(int)

    return Xtr, ytr, Xte, yte

# ============================================================
# METRICS
# ============================================================
def ece_score(y_true, y_prob, n_bins=10, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            mask = (y_prob >= lo) & (y_prob < hi)
        else:
            mask = (y_prob >= lo) & (y_prob <= hi)

        if np.any(mask):
            acc = np.mean(y_pred[mask] == y_true[mask])
            conf = np.mean(y_prob[mask])
            ece += (np.sum(mask) / len(y_true)) * abs(acc - conf)
    return float(ece)

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp + 1e-12)
    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    bal_acc = 0.5 * (specificity + sensitivity)

    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
        "Specificity": float(specificity),
        "BalancedAcc": float(bal_acc),
        "ROC_AUC": float(roc_auc_score(y_true, y_prob)),
        "PR_AUC": float(average_precision_score(y_true, y_prob)),
        "Brier": float(brier_score_loss(y_true, y_prob)),
        "ECE": float(ece_score(y_true, y_prob, threshold=threshold)),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }

# ============================================================
# MODEL FACTORY
# ============================================================
def make_models(seed):
    models = {}

    models["DecisionTree"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", DecisionTreeClassifier(
            max_depth=None,
            random_state=seed
        ))
    ])

    models["RandomForest"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=seed,
            n_jobs=-1
        ))
    ])

    models["GradientBoost"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=seed
        ))
    ])

    models["KNN"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(
            n_neighbors=7,
            weights="distance"
        ))
    ])

    models["SVM"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C=2.0,
            kernel="rbf",
            probability=True,
            random_state=seed
        ))
    ])

    models["MLP"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size="auto",
            learning_rate_init=1e-3,
            max_iter=1000,
            early_stopping=True,
            random_state=seed
        ))
    ])

    if HAS_XGBOOST:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=seed,
                n_jobs=-1
            ))
        ])

    if HAS_CATBOOST:
        models["CatBoost"] = CatBoostClassifier(
            iterations=300,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="AUC",
            verbose=False,
            random_seed=seed
        )

    return models

# ============================================================
# TRAIN + PREDICT WITH TIME SPLIT
# ============================================================
def fit_predict_prob_with_time(model, Xtr, ytr, Xte, algo_name=None):
    # Special handling for CatBoost
    if algo_name == "CatBoost":
        imputer = SimpleImputer(strategy="median")

        t0 = time.time()
        Xtr_imp = imputer.fit_transform(Xtr)
        Xte_imp = imputer.transform(Xte)
        model.fit(Xtr_imp, ytr)
        t1 = time.time()

        t2 = time.time()
        y_prob = model.predict_proba(Xte_imp)[:, 1]
        t3 = time.time()

        return y_prob, (t1 - t0), (t3 - t2)

    # Default sklearn / pipeline flow
    t0 = time.time()
    model.fit(Xtr, ytr)
    t1 = time.time()

    t2 = time.time()
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(Xte)[:, 1]
    else:
        score = model.decision_function(Xte)
        y_prob = 1.0 / (1.0 + np.exp(-score))
    t3 = time.time()

    return y_prob, (t1 - t0), (t3 - t2)

# ============================================================
# RUN ALL
# ============================================================
raw_rows = []

available_note = []
if not HAS_XGBOOST:
    available_note.append("XGBoost not installed -> skipped")
if not HAS_CATBOOST:
    available_note.append("CatBoost not installed -> skipped")

if available_note:
    print("\n".join(available_note))

for k in PCA_LIST:
    print("\n" + "=" * 100)
    print(f"RUNNING CLASSICAL BASELINES | {DS.upper()} | PCA-{k}")
    print("=" * 100)

    Xtr, ytr, Xte, yte = load_train_test(DATA_ROOT, DS, k)

    for seed in SEEDS:
        models = make_models(seed)

        for algo_name, model in models.items():
            y_prob, train_t, test_t = fit_predict_prob_with_time(
                model, Xtr, ytr, Xte, algo_name=algo_name
            )

            m = compute_metrics(yte, y_prob, threshold=THRESH)
            m.update({
                "Dataset": DS,
                "PCA": k,
                "Seed": seed,
                "Algorithm": algo_name,
                "TrainN": int(len(Xtr)),
                "TestN": int(len(Xte)),
                "TrainTime_s": float(train_t),
                "TestTime_s": float(test_t),
                "TotalTime_s": float(train_t + test_t),
            })
            raw_rows.append(m)

            print(
                f"{algo_name:>13} | seed={seed} | "
                f"Acc={m['Accuracy']:.4f} | "
                f"F1={m['F1']:.4f} | "
                f"MCC={m['MCC']:.4f} | "
                f"ROC_AUC={m['ROC_AUC']:.4f} | "
                f"Train={m['TrainTime_s']:.4f}s | "
                f"Test={m['TestTime_s']:.4f}s | "
                f"Total={m['TotalTime_s']:.4f}s"
            )

raw_df = pd.DataFrame(raw_rows)

raw_csv = os.path.join(OUT_DIR, "raw_all_seeds_ml_baselines_balanced_pca12.csv")
raw_df.to_csv(raw_csv, index=False)

# ============================================================
# MEAN / STD / MEAN±STD
# ============================================================
group_cols = ["Dataset", "PCA", "Algorithm"]
ignore_cols = group_cols + ["Seed", "TrainN", "TestN", "TP", "TN", "FP", "FN"]
metric_cols = [c for c in raw_df.columns if c not in ignore_cols]

mean_df = raw_df.groupby(group_cols)[metric_cols].mean().reset_index()
std_df = raw_df.groupby(group_cols)[metric_cols].std().reset_index()

mean_csv = os.path.join(OUT_DIR, "mean_ml_baselines_balanced_pca12.csv")
std_csv  = os.path.join(OUT_DIR, "std_ml_baselines_balanced_pca12.csv")

mean_df.to_csv(mean_csv, index=False)
std_df.to_csv(std_csv, index=False)

pm_df = mean_df.copy()
for c in metric_cols:
    pm_df[c] = mean_df[c].map(lambda x: f"{x:.4f}") + " ± " + std_df[c].map(lambda x: f"{x:.4f}")

pm_csv = os.path.join(OUT_DIR, "meanpmstd_ml_baselines_balanced_pca12.csv")
pm_df.to_csv(pm_csv, index=False)

# ============================================================
# DISPLAY TABLE + HIGHLIGHT BEST CELLS
# ============================================================
cols_show = [
    "Dataset", "PCA", "Algorithm",
    "Accuracy", "Precision", "Recall", "F1", "MCC",
    "ROC_AUC", "PR_AUC", "Brier", "ECE",
    "TrainTime_s", "TestTime_s", "TotalTime_s"
]

disp_pm_df = pm_df[cols_show].copy()
disp_mean_df = mean_df[cols_show].copy()

higher_better = ["Accuracy", "Precision", "Recall", "F1", "MCC", "ROC_AUC", "PR_AUC"]
lower_better = ["Brier", "ECE", "TrainTime_s", "TestTime_s", "TotalTime_s"]

BEST_COLOR = "background-color: #d9ead3;"
BORDER = "1px solid black"

def highlight_best_cells_within_pca(_):
    styles = pd.DataFrame("", index=disp_pm_df.index, columns=disp_pm_df.columns)

    for pca_val in disp_mean_df["PCA"].unique():
        mask = disp_mean_df["PCA"] == pca_val
        sub = disp_mean_df.loc[mask]

        for c in higher_better:
            best = sub[c].max()
            styles.loc[mask & np.isclose(disp_mean_df[c], best), c] = BEST_COLOR

        for c in lower_better:
            best = sub[c].min()
            styles.loc[mask & np.isclose(disp_mean_df[c], best), c] = BEST_COLOR

    return styles

sty = (
    disp_pm_df.style
    .apply(highlight_best_cells_within_pca, axis=None)
    .set_properties(**{
        "text-align": "center",
        "vertical-align": "middle",
        "border": BORDER,
        "font-size": "11pt",
        "font-family": "Times New Roman"
    })
    .set_table_styles([
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("border", BORDER),
                ("width", "100%"),
            ]
        },
        {
            "selector": "th",
            "props": [
                ("border", BORDER),
                ("text-align", "center"),
                ("vertical-align", "middle"),
                ("background-color", "#f2f2f2"),
                ("font-weight", "bold"),
                ("padding", "6px"),
                ("font-family", "Times New Roman"),
                ("font-size", "11pt")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("border", BORDER),
                ("text-align", "center"),
                ("vertical-align", "middle"),
                ("padding", "6px"),
                ("font-family", "Times New Roman"),
                ("font-size", "11pt")
            ]
        },
        {
            "selector": ".row_heading",
            "props": [
                ("border", BORDER),
                ("text-align", "center"),
                ("background-color", "#f2f2f2"),
                ("font-weight", "bold")
            ]
        },
        {
            "selector": ".blank",
            "props": [
                ("border", BORDER),
                ("background-color", "#f2f2f2")
            ]
        }
    ])
)

print("\n=== MEAN ± STD TABLE: Classical ML Baselines | Balanced | PCA-12 ===\n")

try:
    from IPython.display import display
    display(sty)
except Exception:
    print(disp_pm_df.to_string(index=False))

try:
    import openpyxl
    out_xlsx = os.path.join(OUT_DIR, "highlighted_bestcells_meanpmstd_ml_baselines_balanced_pca12.xlsx")
    sty.to_excel(out_xlsx, engine="openpyxl", index=True)
    print("\nHighlighted Excel saved:", out_xlsx)
except Exception as e:
    print("\nExcel export skipped:", e)

try:
    html_path = os.path.join(OUT_DIR, "highlighted_bestcells_meanpmstd_ml_baselines_balanced_pca12.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(sty.to_html())
    print("Styled HTML saved:", html_path)
except Exception as e:
    print("HTML export skipped:", e)

print("\nSaved files:")
print(raw_csv)
print(mean_csv)
print(std_csv)
print(pm_csv)
print("\nDone.")

Results will be saved in:
C:\Users\shanmugam\shan work 1\New folder\new try\output

RUNNING CLASSICAL BASELINES | BALANCED | PCA-12
 DecisionTree | seed=0 | Acc=0.9367 | F1=0.9343 | MCC=0.8757 | ROC_AUC=0.9367 | Train=0.0060s | Test=0.0000s | Total=0.0060s
 RandomForest | seed=0 | Acc=0.9767 | F1=0.9766 | MCC=0.9534 | ROC_AUC=0.9986 | Train=0.4016s | Test=0.0707s | Total=0.4723s
GradientBoost | seed=0 | Acc=0.9633 | F1=0.9632 | MCC=0.9267 | ROC_AUC=0.9960 | Train=0.3294s | Test=0.0063s | Total=0.3357s
          KNN | seed=0 | Acc=0.9600 | F1=0.9608 | MCC=0.9207 | ROC_AUC=0.9924 | Train=0.0043s | Test=0.0040s | Total=0.0083s
          SVM | seed=0 | Acc=0.9833 | F1=0.9832 | MCC=0.9669 | ROC_AUC=0.9996 | Train=0.0163s | Test=0.0020s | Total=0.0183s
          MLP | seed=0 | Acc=0.9467 | F1=0.9467 | MCC=0.8933 | ROC_AUC=0.9868 | Train=0.0422s | Test=0.0013s | Total=0.0435s
      XGBoost | seed=0 | Acc=0.9667 | F1=0.9664 | MCC=0.9334 | ROC_AUC=0.9977 | Train=0.0932s | Test=0.0030s | Total=0

,Dataset,PCA,Algorithm,Accuracy,Precision,Recall,F1,MCC,ROC_AUC,PR_AUC,Brier,ECE,TrainTime_s,TestTime_s,TotalTime_s
0,balanced,12,CatBoost,0.9720 ± 0.0038,0.9720 ± 0.0055,0.9720 ± 0.0030,0.9720 ± 0.0038,0.9440 ± 0.0076,0.9982 ± 0.0002,0.9982 ± 0.0002,0.0177 ± 0.0010,0.4852 ± 0.0042,0.6335 ± 0.0759,0.0014 ± 0.0008,0.6349 ± 0.0767
1,balanced,12,DecisionTree,0.9367 ± 0.0113,0.9556 ± 0.0122,0.9160 ± 0.0203,0.9353 ± 0.0119,0.8743 ± 0.0224,0.9367 ± 0.0113,0.9173 ± 0.0141,0.0633 ± 0.0113,0.5000 ± 0.0000,0.0044 ± 0.0009,0.0002 ± 0.0004,0.0046 ± 0.0009
2,balanced,12,GradientBoost,0.9653 ± 0.0018,0.9666 ± 0.0001,0.9640 ± 0.0037,0.9653 ± 0.0019,0.9307 ± 0.0036,0.9960 ± 0.0001,0.9962 ± 0.0001,0.0254 ± 0.0001,0.4786 ± 0.0012,0.3260 ± 0.0066,0.0019 ± 0.0025,0.3279 ± 0.0078
3,balanced,12,KNN,0.9600 ± 0.0000,0.9423 ± 0.0000,0.9800 ± 0.0000,0.9608 ± 0.0000,0.9207 ± 0.0000,0.9924 ± 0.0000,0.9886 ± 0.0000,0.0294 ± 0.0000,0.4608 ± 0.0000,0.0025 ± 0.0010,0.0032 ± 0.0004,0.0057 ± 0.0015
4,balanced,12,MLP,0.9560 ± 0.0229,0.9809 ± 0.0200,0.9307 ± 0.0480,0.9544 ± 0.0252,0.9143 ± 0.0426,0.9932 ± 0.0061,0.9941 ± 0.0050,0.1011 ± 0.0406,0.4769 ± 0.0187,0.0421 ± 0.0040,0.0011 ± 0.0001,0.0431 ± 0.0040
5,balanced,12,RandomForest,0.9760 ± 0.0015,0.9786 ± 0.0029,0.9733 ± 0.0000,0.9759 ± 0.0015,0.9520 ± 0.0030,0.9984 ± 0.0002,0.9984 ± 0.0002,0.0243 ± 0.0006,0.4796 ± 0.0020,0.3956 ± 0.0035,0.0694 ± 0.0008,0.4650 ± 0.0043
6,balanced,12,SVM,0.9833 ± 0.0000,0.9932 ± 0.0000,0.9733 ± 0.0000,0.9832 ± 0.0000,0.9669 ± 0.0000,0.9996 ± 0.0000,0.9996 ± 0.0000,0.0100 ± 0.0002,0.4989 ± 0.0011,0.0151 ± 0.0010,0.0025 ± 0.0005,0.0176 ± 0.0007
7,balanced,12,XGBoost,0.9713 ± 0.0038,0.9732 ± 0.0002,0.9693 ± 0.0076,0.9713 ± 0.0039,0.9427 ± 0.0076,0.9977 ± 0.0001,0.9978 ± 0.0000,0.0178 ± 0.0006,0.4902 ± 0.0039,0.0746 ± 0.0105,0.0020 ± 0.0007,0.0766 ± 0.0110



Highlighted Excel saved: C:\Users\shanmugam\shan work 1\New folder\new try\output\highlighted_bestcells_meanpmstd_ml_baselines_balanced_pca12.xlsx
Styled HTML saved: C:\Users\shanmugam\shan work 1\New folder\new try\output\highlighted_bestcells_meanpmstd_ml_baselines_balanced_pca12.html

Saved files:
C:\Users\shanmugam\shan work 1\New folder\new try\output\raw_all_seeds_ml_baselines_balanced_pca12.csv
C:\Users\shanmugam\shan work 1\New folder\new try\output\mean_ml_baselines_balanced_pca12.csv
C:\Users\shanmugam\shan work 1\New folder\new try\output\std_ml_baselines_balanced_pca12.csv
C:\Users\shanmugam\shan work 1\New folder\new try\output\meanpmstd_ml_baselines_balanced_pca12.csv

Done.
